# Oturum 5a — All-Atom Trajektori Analizi

**Biyofizik 2026 Kursu · Doç. Dr. Mustafa Tekpınar**

Bugüne kadar hep **girdi** hazırladık. Şimdi çıktıya bakıyoruz:
elimize bir trajektori geldiğinde ona hangi soruları sorarız?

> ℹ️ Kursta uzun simülasyon koşmadığımız için **hazır bir trajektori** kullanıyoruz.


---
## 1 · Kurulum


In [ ]:
%%capture
!apt-get -qq update
!apt-get -qq install -y gromacs
!pip install -q MDAnalysis


In [ ]:
!gmx --version 2>&1 | grep -i 'GROMACS version'


---
## 2 · Trajektoriyi yükle

Kurs deposundaki hazır dosyaları kullanacağız. Alternatif olarak sol paneldeki
📁 simgesinden kendi `md.tpr` / `md.xtc` dosyalarınızı da yükleyebilirsiniz.


In [ ]:
# Kurs deposunu klonla
!git clone -q https://github.com/eygpcr/biyofizik2026-martini.git repo
!ls -la repo/05_analiz/trajektori/


In [ ]:
import os, shutil
src = 'repo/05_analiz/trajektori'
for f in os.listdir(src):
    if f.endswith(('.tpr','.xtc','.gro')):
        shutil.copy(os.path.join(src,f), '.')
!ls -lh *.tpr *.xtc 2>/dev/null || print('Dosyalari sol panelden elle yukleyin')


---
## 3 · Önce PBC'yi düzeltin

**En sık atlanan adım.** Periyodik sınır koşulları yüzünden protein kutunun
kenarından çıkıp öbür taraftan girer. Düzeltmezseniz RMSD saçmalar.


In [ ]:
# 1) Sicramalari kaldir
!echo 0 | gmx trjconv -s md.tpr -f md.xtc -o md_nojump.xtc -pbc nojump

# 2) Proteine gore hizala (once fit grubu: Protein, sonra cikti grubu: System)
!echo -e '1\n0' | gmx trjconv -s md.tpr -f md_nojump.xtc -o md_fit.xtc -fit rot+trans


---
## 4 · RMSD — sistem dengelendi mi?

**Nasıl okunur:** Eğri bir platoya oturmuşsa sistem dengelenmiştir.
Sürekli tırmanıyorsa ya daha uzun koşmalısınız ya da yapı bozuluyordur.


In [ ]:
!echo -e '4\n4' | gmx rms -s md.tpr -f md_fit.xtc -o rmsd.xvg


### `.xvg` dosyalarını çizmek için yardımcı fonksiyon


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def xvg(path):
    """GROMACS .xvg dosyasini numpy dizisi olarak okur."""
    return np.loadtxt(path, comments=['#','@'])

def ciz(path, xlabel, ylabel, baslik, sutun=1):
    d = xvg(path)
    plt.figure(figsize=(8,4))
    plt.plot(d[:,0], d[:,sutun], lw=1.2)
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.title(baslik)
    plt.grid(alpha=.3); plt.tight_layout(); plt.show()
    return d


In [ ]:
d = ciz('rmsd.xvg', 'Zaman (ps)', 'RMSD (nm)', 'Omurga RMSD')
print(f'Son 20% ortalama: {d[int(len(d)*.8):,1].mean():.3f} nm')


> 🔍 **Tartışma:** Referans olarak ilk kareyi mi, kristal yapıyı mı almalıyız?
> Fark eder mi? Analizinizi hangi andan sonra başlatmalısınız?


---
## 5 · RMSF — hangi bölgeler esnek?

**Nasıl okunur:** Loop'lar ve terminaller yüksek, heliks/yaprak düşük çıkar.
Beklenmedik bir yerde yüksek RMSF = ya ilginç bir bulgu ya da kurulum hatası.


In [ ]:
!echo 4 | gmx rmsf -s md.tpr -f md_fit.xtc -o rmsf.xvg -res
d = ciz('rmsf.xvg', 'Rezidu no', 'RMSF (nm)', 'Rezidu bazinda dalgalanma')

en_esnek = d[d[:,1].argsort()[-10:]][::-1]
print('En esnek 10 rezidu:')
for r, v in en_esnek: print(f'  rezidu {int(r):>4}: {v:.3f} nm')


> 🔍 RMSF'i kristal yapının **B-faktörleriyle** karşılaştırın. Uyuşuyorlar mı?


---
## 6 · Rg — protein kompakt kaldı mı?

Ani artış = açılma (unfolding) veya kurulum sorunu.


In [ ]:
!echo 4 | gmx gyrate -s md.tpr -f md_fit.xtc -o gyrate.xvg
ciz('gyrate.xvg', 'Zaman (ps)', 'Rg (nm)', 'Jirasyon yaricapi');


---
## 7 · Hidrojen bağları

> 🎓 **Bunu aklınızda tutun.** 5b'de Martini'ye geçtiğimizde bu analizi
> *yapamayacağız* — çünkü Martini'de hidrojen yok.


In [ ]:
!echo -e '1\n1' | gmx hbond -s md.tpr -f md_fit.xtc -num hbond.xvg
ciz('hbond.xvg', 'Zaman (ps)', 'H-bagi sayisi', 'Protein ici hidrojen baglari');


---
## 8 · Yoğunluk profili *(membranlı sistemler için)*

Membran normali (z) boyunca su, lipid ve proteinin nerede olduğunu gösterir.
Membran kalınlığını buradan okursunuz.


In [ ]:
!echo 0 | gmx density -s md.tpr -f md_fit.xtc -o density.xvg -d Z -sl 100
ciz('density.xvg', 'z (nm)', 'Yogunluk (kg/m3)', 'Membran normali boyunca yogunluk');


---
## ✅ Kontrol listesi — bir trajektoriyi ilk gördüğünüzde

- [ ] PBC düzeltildi mi?
- [ ] Enerji ve sıcaklık kararlı mı?
- [ ] RMSD platoya oturdu mu? Analizi hangi andan sonra yapmalıyım?
- [ ] Sistem çökmüş/patlamış mı? *(görsel kontrol şart)*
- [ ] Membran varsa: alan/lipid ve kalınlık makul mü?

---

## 📚 Kaynaklar

- [GROMACS analiz araçları](https://manual.gromacs.org/current/user-guide/cmdline.html#commands-by-topic)
- [GROMACS tutorials](https://tutorials.gromacs.org/)
- [MDAnalysis](https://www.mdanalysis.org/) — Python'la özel analiz yazmak için

> ⏭️ **Sırada:** aynı analizleri Martini'ye uyguladığımızda ne değişiyor?
> → [`05_analiz/martini/`](https://github.com/eygpcr/biyofizik2026-martini/tree/main/05_analiz/martini)
